[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/090.%20The%20Global%20Sourcing%20Offshoring%20vs.%20Reshoring%20Problem/P90-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 90. The Global Sourcing Offshoring vs. Reshoring Problem
## Tier 7: The Human–AI Symbiotic Partnership

### Purpose
This notebook demonstrates a human-in-the-loop sourcing workflow. An AI component generates and evaluates supplier scenarios; a procurement expert reviews the recommendations, adds strategic context, and participates in monitoring. The computational model is deliberately transparent so that its assumptions and limitations can be examined.

### Inputs and outputs
The example uses a fixed database of five suppliers and three products, including product demand, supplier cost, risk, quality, capacity, sustainability, and innovation attributes. It generates 50 supplier-subset scenarios using the existing random seed. The workflow returns evaluated sourcing allocations, human-review assessments, implementation risks, and monitoring visualizations.

### Scope and modeling assumptions
- A scenario contains a randomly selected subset of suppliers. The allocation rule distributes each product's demand among those suppliers in proportion to weighted supplier scores.
- The allocation rule is a heuristic, not a capacity-constrained mathematical program. Supplier capacity is checked during implementation-risk screening; it does not constrain the allocation.
- The objective combines monetary cost with quantity-weighted risk and quality scores. These components have different units, so the resulting objective is an illustrative index, not a monetary total. Its weights and input scales must be calibrated before operational use.
- Human alignment, confidence, and stakeholder satisfaction are transparent proxies calculated from the supplied attributes. They do not measure actual human judgment, trust, or business performance.
- The supplier and product data are illustrative. The notebook does not establish real-world performance claims or validate the simulated indicators against historical outcomes.

### Workflow
1. Synthesize machine-readable indicators and human-provided strategic context.
2. Generate supplier-subset scenarios and calculate an allocation for each.
3. Rank scenarios by the stated objective and have the human expert review leading candidates.
4. Re-evaluate selected candidates and generate explanations.
5. Identify implementation risks and prepare stakeholder communication points.
6. Demonstrate repeated monitoring using explicitly defined proxy measures.

### Mathematical model used by the allocation rule
For product \(p\) and supplier \(s\), the code first calculates a positive score

$$
q_s = \alpha\frac{1}{c_s} + \beta\frac{1}{1+r_s} + \gamma u_s,
$$

then allocates product demand in proportion to that supplier score:

$$
x_{ps} = d_p\frac{q_s}{\sum_{j\in S}q_j},
\qquad
\sum_{s\in S}x_{ps}=d_p.
$$

Here, \(S\) is the supplier subset in a scenario; \(d_p\) is demand for product \(p\); \(c_s\), \(r_s\), and \(u_s\) are supplier unit cost, risk score, and quality score; and \((\alpha,\beta,\gamma)=(0.6,0.3,0.1)\). In the code, these quantities are represented by `scenario['suppliers']`, `scenario['demands']`, `supplier_costs`, `supplier_risks`, `supplier_qualities`, and the corresponding attributes of `OptimizationEngine`.

The scenario objective is

$$
J = \alpha C + \beta R - \gamma U,
\qquad
C=\sum_{p\in P}\sum_{s\in S}c_sx_{ps},
\quad
R=\sum_{p\in P}\sum_{s\in S}r_sx_{ps},
\quad
U=\sum_{p\in P}\sum_{s\in S}u_sx_{ps}.
$$

The workflow selects the scenario with the smallest \(J\). Thus, cost and risk increase the objective, while quality reduces it. This weighted index is used to rank scenarios; the proportional allocation rule itself does not solve an optimization problem over all possible allocations.

### Human review and confidence
The expert's alignment proxy is a weighted average of supplier attributes over the recommended allocation. Sustainability is represented by ESG score, cost alignment by a normalized inverse-cost score, risk management by \(1-r_s\), and innovation by innovation score. The confidence proxy combines strategic alignment, keyword-based expertise relevance, and an ESG-based ethics proxy. These are teaching devices, not validated measures of expertise or ethical impact.

AI confidence is based on the best objective relative to the range of generated scenario objectives. It is a ranking-separation indicator, not a probability that a recommendation is correct. Repeated or tied scenario values therefore require cautious interpretation.

### Computational scale
For \(N\) generated scenarios, \(P\) products, and at most \(S\) suppliers per scenario, generating and evaluating allocations takes \(O(NPS)\) time. The stored scenario results take \(O(NPS)\) space. This scaling describes the notebook's enumeration of sampled supplier subsets, not an exhaustive search over every possible allocation.

In [ ]:
from collections import deque
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# The fixed seeds make the sampled supplier subsets and simulated monitoring reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
class HumanExpert:
    """Represent a procurement expert and transparent review proxies."""

    def __init__(self, name, expertise_areas, strategic_priorities):
        self.name = name
        self.expertise_areas = expertise_areas
        self.strategic_priorities = strategic_priorities
        self.experience_years = 15
        self.confidence_threshold = 0.8
        self.decision_history = []
        self.preference_patterns = {}
        self.trust_level = 0.5
        self.communication_style = 'analytical'
        self.explanation_preference = 'detailed'

    def evaluate_ai_recommendation(self, recommendation, context):
        """Score a recommendation using declared priorities and supplier data."""
        alignment_score = self._check_strategic_alignment(recommendation, context)
        expertise_score = self._evaluate_expertise_relevance(recommendation)
        ethical_score = self._evaluate_ethical_considerations(recommendation)
        confidence = (
            0.4 * alignment_score
            + 0.3 * expertise_score
            + 0.3 * ethical_score
        )

        if confidence >= self.confidence_threshold:
            acceptance = 'accept'
            modifications = []
            reasoning = 'The proxy scores meet the stated review threshold.'
        elif confidence >= 0.6:
            acceptance = 'modify'
            modifications = self._suggest_modifications()
            reasoning = 'The recommendation merits review and possible refinement.'
        else:
            acceptance = 'reject'
            modifications = []
            reasoning = 'The proxy scores indicate insufficient alignment for acceptance.'

        evaluation = {
            'acceptance': acceptance,
            'modifications': modifications,
            'confidence': confidence,
            'reasoning': reasoning,
            'strategic_alignment': alignment_score,
            'expertise_relevance': expertise_score,
            'ethical_proxy': ethical_score,
        }
        self.decision_history.append(evaluation)
        return evaluation

    def _check_strategic_alignment(self, recommendation, context):
        """Aggregate supplier attributes according to declared priority weights."""
        del context  # Context is retained in the interface for review extensions.
        total_quantity = sum(
            sum(product_allocation.values())
            for product_allocation in recommendation['allocation'].values()
        )
        if total_quantity <= 0:
            return 0.0

        supplier_weights = {}
        for supplier in recommendation['suppliers']:
            quantity = sum(
                product_allocation.get(supplier, 0.0)
                for product_allocation in recommendation['allocation'].values()
            )
            supplier_weights[supplier] = quantity / total_quantity

        attributes = {
            'sustainability': lambda data: data['esg_score'],
            'cost_optimization': lambda data: 1.0 - (
                data['cost'] - 180.0
            ) / (320.0 - 180.0),
            'risk_management': lambda data: 1.0 - data['risk'],
            'innovation': lambda data: data['innovation_score'],
        }

        alignment = 0.0
        for priority, priority_weight in self.strategic_priorities.items():
            attribute = attributes.get(priority)
            if attribute is None:
                continue
            supplier_average = sum(
                supplier_weights[supplier] * attribute(data)
                for supplier, data in recommendation['suppliers'].items()
            )
            alignment += priority_weight * supplier_average
        return min(max(alignment, 0.0), 1.0)

    def _evaluate_expertise_relevance(self, recommendation):
        """Measure whether the recommendation contains declared expertise topics."""
        searchable_text = ' '.join(
            str(value)
            for value in (
                recommendation['suppliers'],
                recommendation.get('context', {}),
            )
        ).lower()
        if not self.expertise_areas:
            return 0.0
        matches = sum(
            area.lower() in searchable_text
            for area in self.expertise_areas
        )
        return matches / len(self.expertise_areas)

    def _evaluate_ethical_considerations(self, recommendation):
        """Use the allocation-weighted ESG score as a limited ethics proxy."""
        total_quantity = sum(
            sum(product_allocation.values())
            for product_allocation in recommendation['allocation'].values()
        )
        if total_quantity <= 0:
            return 0.0
        weighted_esg = 0.0
        for supplier, data in recommendation['suppliers'].items():
            quantity = sum(
                product_allocation.get(supplier, 0.0)
                for product_allocation in recommendation['allocation'].values()
            )
            weighted_esg += quantity * data['esg_score']
        return weighted_esg / total_quantity

    def _suggest_modifications(self):
        """Return review topics rather than silently altering model parameters."""
        modifications = []
        if 'sustainability' in self.expertise_areas:
            modifications.append('Review ESG measures and sustainability reporting.')
        if 'risk_management' in self.expertise_areas:
            modifications.append('Review disruption risks and mitigation plans.')
        if 'cost_optimization' in self.expertise_areas:
            modifications.append('Review total cost assumptions and sensitivity.')
        return modifications


class OptimizationEngine:
    """Apply the notebook's weighted-score proportional allocation rule."""

    def __init__(self):
        self.alpha = 0.6
        self.beta = 0.3
        self.gamma = 0.1

    def optimize(self, scenario):
        """Allocate each product's demand in proportion to supplier scores."""
        allocation = {}
        for product in scenario['products']:
            demand = scenario['demands'][product]
            scores = {}
            for supplier in scenario['suppliers']:
                cost_score = 1.0 / scenario['supplier_costs'][supplier]
                risk_score = 1.0 / (1.0 + scenario['supplier_risks'][supplier])
                quality_score = scenario['supplier_qualities'][supplier]
                scores[supplier] = (
                    self.alpha * cost_score
                    + self.beta * risk_score
                    + self.gamma * quality_score
                )

            total_score = sum(scores.values())
            if total_score <= 0:
                raise ValueError('Supplier scores must have a positive sum.')
            allocation[product] = {
                supplier: demand * score / total_score
                for supplier, score in scores.items()
            }

        total_cost = 0.0
        total_risk = 0.0
        total_quality = 0.0
        for product, product_allocation in allocation.items():
            for supplier, quantity in product_allocation.items():
                total_cost += scenario['supplier_costs'][supplier] * quantity
                total_risk += scenario['supplier_risks'][supplier] * quantity
                total_quality += scenario['supplier_qualities'][supplier] * quantity

        objective_value = (
            self.alpha * total_cost
            + self.beta * total_risk
            - self.gamma * total_quality
        )
        result = {
            'id': scenario['id'],
            'products': scenario['products'],
            'demands': scenario['demands'],
            'suppliers': scenario['suppliers'],
            'supplier_costs': scenario['supplier_costs'],
            'supplier_risks': scenario['supplier_risks'],
            'supplier_qualities': scenario['supplier_qualities'],
            'context': scenario['context'],
            'allocation': allocation,
            'objective_value': objective_value,
            'cost_components': {
                'total_cost': total_cost,
                'total_risk': total_risk,
                'total_quality': total_quality,
            },
        }

        # These checks make the demand balance and objective accounting explicit.
        for product in scenario['products']:
            allocated = sum(allocation[product].values())
            assert math.isclose(
                allocated, scenario['demands'][product], rel_tol=1e-9
            )
        recomputed_objective = (
            self.alpha * total_cost
            + self.beta * total_risk
            - self.gamma * total_quality
        )
        assert math.isclose(
            objective_value, recomputed_objective, rel_tol=1e-12, abs_tol=1e-9
        )
        return result


class ScenarioGenerator:
    """Sample supplier subsets from the illustrative supplier database."""

    def __init__(self):
        self.supplier_database = self._create_supplier_database()
        self.product_database = self._create_product_database()

    def _create_supplier_database(self):
        """Define supplier attributes used by allocation and review."""
        return {
            'TechnoGreen_Netherlands': {
                'location': 'Netherlands', 'cost': 280, 'risk': 0.2,
                'quality': 0.92, 'capacity': 40000, 'esg_score': 0.95,
                'innovation_score': 0.88,
            },
            'CleanCell_SouthKorea': {
                'location': 'South Korea', 'cost': 320, 'risk': 0.3,
                'quality': 0.95, 'capacity': 35000, 'esg_score': 0.93,
                'innovation_score': 0.95,
            },
            'LocalBatt_Texas': {
                'location': 'Texas, USA', 'cost': 240, 'risk': 0.1,
                'quality': 0.90, 'capacity': 20000, 'esg_score': 0.91,
                'innovation_score': 0.75,
            },
            'China_Budget': {
                'location': 'China', 'cost': 180, 'risk': 0.8,
                'quality': 0.70, 'capacity': 60000, 'esg_score': 0.65,
                'innovation_score': 0.60,
            },
            'Vietnam_Cost': {
                'location': 'Vietnam', 'cost': 200, 'risk': 0.5,
                'quality': 0.85, 'capacity': 30000, 'esg_score': 0.78,
                'innovation_score': 0.70,
            },
        }

    def _create_product_database(self):
        """Define illustrative product demands and descriptive attributes."""
        return {
            'Battery_Tech': {
                'demand': 12000, 'criticality': 'high',
                'innovation_level': 'high',
            },
            'Solar_Panel': {
                'demand': 8000, 'criticality': 'medium',
                'innovation_level': 'medium',
            },
            'Wind_Turbine': {
                'demand': 2000, 'criticality': 'high',
                'innovation_level': 'high',
            },
        }

    def generate_scenarios(self, context, num_scenarios=50):
        """Sample three- or four-supplier subsets and package their data."""
        scenarios = []
        supplier_names = list(self.supplier_database)
        for scenario_id in range(num_scenarios):
            selected_count = np.random.randint(3, 5)
            selected_suppliers = np.random.choice(
                supplier_names, size=selected_count, replace=False
            )
            suppliers = {
                name: self.supplier_database[name]
                for name in selected_suppliers
            }
            scenarios.append({
                'id': scenario_id,
                'products': list(self.product_database),
                'demands': {
                    product: details['demand']
                    for product, details in self.product_database.items()
                },
                'suppliers': suppliers,
                'supplier_costs': {
                    name: data['cost'] for name, data in suppliers.items()
                },
                'supplier_risks': {
                    name: data['risk'] for name, data in suppliers.items()
                },
                'supplier_qualities': {
                    name: data['quality'] for name, data in suppliers.items()
                },
                'context': context,
            })
        return scenarios


class ExplanationGenerator:
    """Create explanations directly from the calculated allocation and costs."""

    def generate_explanation(self, scenario, context, user_preferences=None):
        """Summarize the recommendation without asserting unmeasured benefits."""
        del user_preferences  # Kept in the interface for the collaborative workflow.
        components = scenario['cost_components']
        lines = [
            f"Recommendation for scenario {scenario['id']}",
            f"Suppliers in scenario: {len(scenario['suppliers'])}",
            f"Weighted objective index: {scenario['objective_value']:,.2f}",
            f"Total variable cost: ${components['total_cost']:,.2f}",
            f"Quantity-weighted risk component: {components['total_risk']:,.2f}",
            f"Quantity-weighted quality component: {components['total_quality']:,.2f}",
            f"Market condition: {context.get('market_condition', 'unspecified')}",
            '',
            'Allocation by product and supplier:',
        ]
        for product, product_allocation in scenario['allocation'].items():
            lines.append(f"  {product} (demand {scenario['demands'][product]:,.0f}):")
            for supplier, quantity in product_allocation.items():
                lines.append(f"    {supplier}: {quantity:,.1f} units")
        lines.extend([
            '',
            'Interpretation: the objective is a weighted index, not a monetary total.',
            'Capacity is screened separately and is not enforced in this allocation.',
        ])
        return '\n'.join(lines)


class ExplainableAI:
    """Generate, rank, and explain sampled sourcing scenarios."""

    def __init__(self, model_name='ARIA'):
        self.model_name = model_name
        self.confidence_threshold = 0.8
        self.explanation_level = 'detailed'
        self.interaction_history = deque(maxlen=1000)
        self.user_preferences = {}
        self.trust_metrics = {
            'accuracy': 0.0,
            'reliability': 0.0,
            'transparency': 0.0,
        }
        self.optimization_engine = OptimizationEngine()
        self.scenario_generator = ScenarioGenerator()
        self.explanation_generator = ExplanationGenerator()

    def generate_sourcing_recommendation(self, problem_context, human_preferences=None):
        """Evaluate sampled scenarios and return the lowest-index candidate."""
        scenarios = self.scenario_generator.generate_scenarios(
            problem_context, num_scenarios=50
        )
        optimized_scenarios = [
            self.optimization_engine.optimize(scenario)
            for scenario in scenarios
        ]
        ranked_scenarios = sorted(
            optimized_scenarios, key=lambda item: item['objective_value']
        )
        best_scenario = ranked_scenarios[0]
        best_scenario['explanation'] = (
            self.explanation_generator.generate_explanation(
                best_scenario, problem_context, human_preferences
            )
        )
        confidence = self._calculate_confidence(ranked_scenarios)
        return {
            'scenario': best_scenario,
            'explanation': best_scenario['explanation'],
            'confidence': confidence,
            'alternatives': ranked_scenarios[:3],
            'ranked_scenarios': ranked_scenarios,
        }

    def _calculate_confidence(self, scenarios):
        """Return a normalized ranking-separation proxy, not a probability."""
        if len(scenarios) < 2:
            return 0.5
        values = [scenario['objective_value'] for scenario in scenarios]
        minimum = min(values)
        maximum = max(values)
        if math.isclose(minimum, maximum, rel_tol=1e-12, abs_tol=1e-12):
            return 0.5
        mean_value = float(np.mean(values))
        confidence = (mean_value - minimum) / (maximum - minimum)
        return min(max(confidence, 0.0), 1.0)


print('Human-AI sourcing framework initialized.')

In [ ]:
print('=== HUMAN-AI SOURCING PARTNERSHIP ===')
print('Initializing the expert review and explainable recommendation workflow.\n')

sarah = HumanExpert(
    name='Sarah Chen',
    expertise_areas=['sustainability', 'innovation', 'risk_management'],
    strategic_priorities={
        'sustainability': 0.4,
        'cost_optimization': 0.3,
        'risk_management': 0.2,
        'innovation': 0.1,
    },
)
aria = ExplainableAI('ARIA')

print(f'Human expert: {sarah.name}')
print(f'AI system: {aria.model_name}')
print(f'Expertise areas: {", ".join(sarah.expertise_areas)}')
print(f'Strategic priorities: {sarah.strategic_priorities}')
print(f'Initial human trust proxy: {sarah.trust_level:.2f}')
print(f'Initial AI monitoring metrics: {aria.trust_metrics}')

In [ ]:
problem_context = {
    'market_condition': 'dynamic',
    'time_horizon': '24 months',
    'growth_phase': 'expansion',
    'strategic_priorities': sarah.strategic_priorities,
    'current_challenges': [
        'Geopolitical tensions affecting traditional suppliers',
        'Quality issues with current battery suppliers',
        'Need for innovation in battery technology',
        'ESG requirements from customers',
        'Supply chain resilience concerns',
    ],
}

human_preferences = {
    'sustainability_weight': 0.4,
    'innovation_priority': 'high',
    'risk_tolerance': 'medium',
    'cost_sensitivity': 'medium',
    'supplier_preferences': {
        'geographic_diversification': 'high',
        'local_sourcing': 'preferred',
        'supplier_relationships': 'important',
    },
}

print('Problem context:')
for key, value in problem_context.items():
    print(f'  {key}: {value}')
print('\nHuman preferences:')
for key, value in human_preferences.items():
    print(f'  {key}: {value}')

## Phase 1: Data Synthesis

The AI-readable indicators and the expert's strategic context serve different purposes. Numeric indicators are inputs to the scenario model; qualitative context supports interpretation and human review. Combining them in one context object does not mean that every qualitative statement is automatically converted into a model constraint.

In [ ]:
print('=== PHASE 1: DATA SYNTHESIS ===')

ai_data = {
    'market_trends': {
        'demand_growth': 0.15,
        'price_volatility': 0.08,
        'competitor_actions': [
            'price_war', 'capacity_expansion', 'service_improvement'
        ],
    },
    'supplier_performance': {
        'capacity_utilization': 0.85,
        'quality_trends': 'improving',
        'cost_trends': 'stable',
    },
    'risk_indicators': {
        'geopolitical_risk': 'elevated',
        'supply_disruption_probability': 0.25,
        'quality_incidents': 'increasing',
    },
}

human_insights = {
    'market_intelligence': 'Customer demand is increasing for sustainable products.',
    'competitive_landscape': 'New entrants are appearing in battery technology.',
    'regulatory_changes': 'ESG regulations are becoming stricter.',
    'stakeholder_expectations': 'The board prioritizes sustainability alongside cost.',
    'innovation_requirements': 'Next-generation battery technology is important.',
}

combined_context = {
    'ai_data': ai_data,
    'human_insights': human_insights,
    'problem_context': problem_context,
    'human_preferences': human_preferences,
}

print('Human strategic insights:')
for insight, value in human_insights.items():
    print(f'  {insight}: {value}')
print(f'\nCombined context contains {len(combined_context)} components.')

## Phase 2: Scenario Generation and Human Review

The AI samples 50 supplier subsets, applies the same proportional allocation rule to each, and ranks them by the objective \(J\). The three lowest-index candidates are presented for expert review. The review scores are proxies derived from the model's supplier attributes; they are not independent evidence about real supplier performance.

A candidate marked for modification receives explicit review topics. Those topics are not silently converted into numerical model changes: a real implementation would require the expert to specify an auditable parameter or allocation change.

In [ ]:
print('=== PHASE 2: SCENARIO GENERATION ===')

ai_recommendation = aria.generate_sourcing_recommendation(
    combined_context, human_preferences
)
ranked_scenarios = ai_recommendation['ranked_scenarios']
review_candidates = ai_recommendation['alternatives']

print(f'Sampled and evaluated scenarios: {len(ranked_scenarios)}')
print(f'Candidates presented for review: {len(review_candidates)}')
print(
    'Best-scenario ranking-separation proxy: '
    f"{ai_recommendation['confidence']:.3f}"
)
print('This proxy is not a probability of correctness.\n')

selected_scenarios = []
modified_scenarios = []
rejected_scenarios = []
review_results = []

for rank, scenario in enumerate(review_candidates, start=1):
    evaluation = sarah.evaluate_ai_recommendation(scenario, combined_context)
    review_results.append((scenario, evaluation))
    print(
        f"Candidate {rank} (scenario {scenario['id']}): "
        f"{evaluation['acceptance'].upper()}"
    )
    print(f"  Strategic alignment proxy: {evaluation['strategic_alignment']:.3f}")
    print(f"  Review confidence proxy: {evaluation['confidence']:.3f}")
    print(f"  Reasoning: {evaluation['reasoning']}")

    if evaluation['acceptance'] == 'accept':
        selected_scenarios.append(scenario)
    elif evaluation['acceptance'] == 'modify':
        modified_scenarios.append(scenario)
        print(f"  Review topics: {evaluation['modifications']}")
    else:
        rejected_scenarios.append(scenario)

# Keep the best reviewed candidate available for explicit human consideration if
# every proxy score falls below the review thresholds; this does not auto-approve it.
if not selected_scenarios and not modified_scenarios and review_results:
    fallback_scenario, fallback_evaluation = max(
        review_results,
        key=lambda item: item[1]['strategic_alignment'],
    )
    modified_scenarios.append(fallback_scenario)
    print(
        '\nNo candidate met an automatic review threshold. '
        f"Scenario {fallback_scenario['id']} is retained for manual review."
    )
    print(f"Manual-review reasoning: {fallback_evaluation['reasoning']}")

final_scenarios = selected_scenarios + modified_scenarios
print('\nReview totals:')
print(f'  Accepted: {len(selected_scenarios)}')
print(f'  Retained for modification or manual review: {len(modified_scenarios)}')
print(f'  Rejected: {len(rejected_scenarios)}')

## Phase 3: Solution Development

The selected candidates are passed through the same allocation rule so that the displayed solution and its explanation use the same calculations. This step does not claim that a textual review topic has changed the numerical model. The expert must define and approve any such change before it can be represented computationally.

In [ ]:
print('=== PHASE 3: SOLUTION DEVELOPMENT ===')

final_scenarios_optimized = []
final_explanations = []
final_confidences = []

for position, reviewed_scenario in enumerate(final_scenarios, start=1):
    optimized_scenario = aria.optimization_engine.optimize(reviewed_scenario)
    explanation = aria.explanation_generator.generate_explanation(
        optimized_scenario, combined_context, human_preferences
    )
    evaluation = sarah.evaluate_ai_recommendation(
        optimized_scenario, combined_context
    )
    optimized_scenario['human_review'] = evaluation
    optimized_scenario['confidence'] = evaluation['confidence']

    final_scenarios_optimized.append(optimized_scenario)
    final_explanations.append(explanation)
    final_confidences.append(evaluation['confidence'])

    print(
        f"Candidate {position}/{len(final_scenarios)} "
        f"(scenario {optimized_scenario['id']})"
    )
    print(f"  Weighted objective index: {optimized_scenario['objective_value']:,.2f}")
    print(f"  Human review proxy: {evaluation['confidence']:.3f}")

if final_scenarios_optimized:
    final_scenarios_optimized.sort(
        key=lambda item: item['objective_value']
    )
    best_reviewed_scenario = final_scenarios_optimized[0]
    print('\nExplanation for the leading reviewed candidate:')
    print(
        aria.explanation_generator.generate_explanation(
            best_reviewed_scenario, combined_context, human_preferences
        )
    )
else:
    best_reviewed_scenario = None
    print('No candidate is available for solution development.')

## Phase 4: Implementation Planning

Implementation planning checks the calculated allocation against supplier capacities and flags selected risk indicators. These checks inform human review; they do not feed back into the allocation rule. The capacity check aggregates quantities across products for each supplier, because capacity is defined at the supplier level in this example.

In [ ]:
def identify_implementation_risks(scenario):
    """Flag capacity, quality, and location risks for human review."""
    risks = []
    supplier_quantities = {
        supplier: sum(
            product_allocation.get(supplier, 0.0)
            for product_allocation in scenario['allocation'].values()
        )
        for supplier in scenario['suppliers']
    }

    for supplier, quantity in supplier_quantities.items():
        supplier_data = scenario['suppliers'][supplier]
        capacity = supplier_data['capacity']
        if quantity > 0.9 * capacity:
            risks.append(
                f'Capacity utilization exceeds 90% at {supplier}.'
            )
        if supplier_data['quality'] < 0.8:
            risks.append(f'Quality score is below 0.8 at {supplier}.')
        if supplier_data['location'] in {'China', 'Russia', 'Iran'}:
            risks.append(f'Location requires geopolitical-risk review: {supplier}.')
    return risks


def develop_stakeholder_strategy(scenario):
    """Prepare concise stakeholder communication points from calculated values."""
    components = scenario['cost_components']
    return {
        'executive_summary': (
            f"Scenario {scenario['id']} allocates demand across "
            f"{len(scenario['suppliers'])} suppliers."
        ),
        'key_metrics': [
            f"Variable cost: ${components['total_cost']:,.2f}",
            f"Risk component: {components['total_risk']:,.2f}",
            f"Quality component: {components['total_quality']:,.2f}",
        ],
        'discussion_topics': [
            'Explain that the objective is a weighted index, not monetary value.',
            'Review capacity and geographic risk flags before implementation.',
            'Confirm whether the illustrative supplier data reflect current evidence.',
        ],
        'communication_approach': 'collaborative',
        'planning_horizon_months': 6,
    }


def create_implementation_timeline(scenario):
    """Return the illustrative planning horizon used in this example."""
    del scenario
    return 6


implementation_plans = []
stakeholder_strategies = []
print('=== PHASE 4: IMPLEMENTATION PLANNING ===')

for position, scenario in enumerate(final_scenarios_optimized, start=1):
    risks = identify_implementation_risks(scenario)
    strategy = develop_stakeholder_strategy(scenario)
    plan = {
        'scenario': scenario,
        'risks': risks,
        'strategy': strategy,
        'timeline_months': create_implementation_timeline(scenario),
    }
    implementation_plans.append(plan)
    stakeholder_strategies.append(strategy)

    print(f"Scenario {position} (ID {scenario['id']}):")
    print(f'  Implementation risks flagged: {len(risks)}')
    for risk in risks:
        print(f'    - {risk}')
    print(f"  Planning horizon: {plan['timeline_months']} months")

if not implementation_plans:
    print('No implementation plan was created because no candidate was retained.')

## Phase 5: Joint Monitoring

The monitoring demonstration revisits the retained recommendations over six illustrative months, with four review cycles per month. It records the expert-review proxy, the current trust proxy, a satisfaction proxy derived from strategic alignment, and measured code execution time. Repeated cycles over fixed illustrative data are not evidence of real-world learning, decision improvement, stakeholder trust, or business impact.

In [ ]:
def simulate_collaboration(human, ai, context, scenarios):
    """Record one proxy-based review cycle for a retained scenario."""
    if not scenarios:
        return None

    scenario_index = len(ai.interaction_history) % len(scenarios)
    scenario = scenarios[scenario_index]
    start_time = time.perf_counter()
    evaluation = human.evaluate_ai_recommendation(scenario, context)
    elapsed_seconds = time.perf_counter() - start_time

    # Trust changes modestly toward the current review proxy; this is a
    # transparent illustrative update rule, not a learned or validated model.
    human.trust_level += 0.05 * (evaluation['confidence'] - human.trust_level)
    ai.interaction_history.append({
        'scenario_id': scenario['id'],
        'evaluation': evaluation,
        'elapsed_seconds': elapsed_seconds,
    })

    return {
        'decision_quality_proxy': evaluation['confidence'],
        'processing_time_seconds': elapsed_seconds,
        'trust_proxy': human.trust_level,
        'stakeholder_satisfaction_proxy': evaluation['strategic_alignment'],
        'ai_alignment_proxy': evaluation['strategic_alignment'],
    }


monitoring_data = {
    'decision_quality_proxy': [],
    'processing_time_seconds': [],
    'trust_proxy': [],
    'stakeholder_satisfaction_proxy': [],
    'ai_alignment_proxy': [],
}
monitoring_months = 6
steps_per_month = 4

print('=== PHASE 5: JOINT MONITORING ===')
if final_scenarios_optimized:
    for month in range(monitoring_months):
        month_results = []
        for _ in range(steps_per_month):
            result = simulate_collaboration(
                sarah, aria, combined_context, final_scenarios_optimized
            )
            month_results.append(result)
            for metric, value in result.items():
                monitoring_data[metric].append(value)

        print(f'Month {month + 1}/{monitoring_months}:')
        print(
            '  Mean review-confidence proxy: '
            f"{np.mean([item['decision_quality_proxy'] for item in month_results]):.3f}"
        )
        print(
            '  Mean processing time: '
            f"{np.mean([item['processing_time_seconds'] for item in month_results]):.6f} seconds"
        )
        print(f'  Trust proxy at month end: {sarah.trust_level:.3f}')
        print(
            '  Mean alignment proxy: '
            f"{np.mean([item['stakeholder_satisfaction_proxy'] for item in month_results]):.3f}"
        )

    aria.trust_metrics['accuracy'] = float(
        np.mean(monitoring_data['decision_quality_proxy'])
    )
    aria.trust_metrics['reliability'] = float(
        np.mean(monitoring_data['ai_alignment_proxy'])
    )
    aria.trust_metrics['transparency'] = 1.0

    print('\nMonitoring summary:')
    print(f'  Review cycles recorded: {len(monitoring_data["trust_proxy"])}')
    print(f'  Final human trust proxy: {sarah.trust_level:.3f}')
    print(f'  AI monitoring proxies: {aria.trust_metrics}')
else:
    print('Monitoring was not run because no candidate was retained.')

In [ ]:
if monitoring_data['trust_proxy']:
    months = np.arange(1, len(monitoring_data['trust_proxy']) + 1)
    month_labels = np.ceil(months / steps_per_month).astype(int)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        'Human–AI Sourcing Workflow: Illustrative Monitoring Proxies',
        fontsize=15,
        fontweight='bold',
    )

    axes[0, 0].plot(
        months, monitoring_data['trust_proxy'], marker='o', label='Trust proxy'
    )
    axes[0, 0].set_title('Trust proxy by review cycle')
    axes[0, 0].set_xlabel('Review cycle')
    axes[0, 0].set_ylabel('Score (0–1)')
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(
        months,
        monitoring_data['decision_quality_proxy'],
        marker='o',
        label='Review-confidence proxy',
    )
    axes[0, 1].set_title('Review-confidence proxy by cycle')
    axes[0, 1].set_xlabel('Review cycle')
    axes[0, 1].set_ylabel('Score (0–1)')
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(
        months,
        monitoring_data['stakeholder_satisfaction_proxy'],
        marker='o',
        label='Strategic-alignment proxy',
    )
    axes[1, 0].set_title('Alignment proxy by review cycle')
    axes[1, 0].set_xlabel('Review cycle')
    axes[1, 0].set_ylabel('Score (0–1)')
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    axes[1, 1].plot(
        months,
        monitoring_data['processing_time_seconds'],
        marker='o',
        label='Measured evaluation time',
    )
    axes[1, 1].set_title('Evaluation runtime by review cycle')
    axes[1, 1].set_xlabel('Review cycle')
    axes[1, 1].set_ylabel('Elapsed time (seconds)')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()

    for axis in axes.flat:
        axis.set_xticks(months[::steps_per_month])
        axis.set_xticklabels(month_labels[::steps_per_month])
        axis.set_xlabel('Illustrative month')

    plt.tight_layout()
    plt.show()

    if best_reviewed_scenario is not None:
        supplier_totals = {
            supplier: sum(
                product_allocation.get(supplier, 0.0)
                for product_allocation in best_reviewed_scenario['allocation'].values()
            )
            for supplier in best_reviewed_scenario['suppliers']
        }
        allocation_frame = pd.DataFrame(
            {
                'Supplier': list(supplier_totals),
                'Allocated quantity (units)': list(supplier_totals.values()),
            }
        ).sort_values('Allocated quantity (units)', ascending=False)

        plt.figure(figsize=(10, 5))
        sns.barplot(
            data=allocation_frame,
            x='Supplier',
            y='Allocated quantity (units)',
            hue='Supplier',
            legend=False,
        )
        plt.title(
            f"Supplier allocation in reviewed scenario {best_reviewed_scenario['id']}"
        )
        plt.xlabel('Supplier')
        plt.ylabel('Allocated quantity (units)')
        plt.xticks(rotation=25, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('No monitoring records are available to visualize.')

## Interpreting the Human–AI Partnership

### Complementary roles
The numerical workflow can consistently process the supplied attributes and compare sampled supplier subsets. A human expert can contribute strategic context, question data quality, and decide whether a recommendation is acceptable. In this notebook, those human judgments are represented only by transparent scoring proxies; the code does not replace actual expert review.

### Trust and explainability
An explanation is useful when a reviewer can trace its figures to the allocation and objective components. Trust itself cannot be inferred from a model score. The trust series in this demonstration follows a stated illustrative update rule and must not be interpreted as empirical evidence of trust development.

### Appropriate use
This workflow is a teaching example for strategic sourcing discussions where cost, risk, quality, sustainability, and innovation matter. Before operational use, decision-makers would need validated data, calibrated objective scales, explicit capacity and feasibility constraints, and documented human approval. Those additions would change the scope of this example and are not implemented here.